# Session 21: Text as data and meaning as numbers

You work on the 800 Lakeside Mart customer messages in `feedback.csv`. Rhythm for every step: **predict**, run, **read** the output, **change** one thing. A line that starts with OK means your number matches. A line that starts with NOT YET means finish that cell. Data is fictional and generated.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
# Helper: returns nan instead of stopping when your answer is not ready yet. You do not need to read it.
def safe(f):
    try:
        return f()
    except Exception:
        return float("nan")

---
## Step 1: Load the messages and look first

Text is a variable like any other. Load the file and count the labels before any cleaning.

In [ ]:
import os, re
import numpy as np
import pandas as pd

os.makedirs('keep', exist_ok=True)   # the folder where you save what you keep
fb = pd.read_csv('feedback.csv')
print(fb.shape)
fb.head()

**PREDICT first.** What share of the 800 messages has the sentiment negative: about a fifth, a half or four fifths?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
n_negative = None  # count the rows where sentiment is 'negative'

In [ ]:
majority_share = None  # n_negative divided by the number of messages: the majority-class baseline

In [ ]:
check('negative messages', safe(lambda: n_negative), 405, 0)
check('majority-class baseline share', safe(lambda: majority_share), 0.5062, 0.001)

---
## Step 2: Text preprocessing: lowercase, regex cleaning, tokens, stop words

Messages open with Hello. or Feedback: and close with Please reply. A regular expression (regex) removes them. Do not remove the word not.

In [ ]:
# Patterns for the opener and closer lines (already written for you)
OPEN = '^(feedback:|to the manager:|hello\\.|on friday evening\\.|yesterday i shopped at [a-z ]+\\.)\\s*'
CLOSE = '(?<=[.?!])\\s+(please reply\\.|keep it up\\.|please fix this\\.|will recommend\\.|do better\\.|i am not happy\\.|thank you\\.|this has happened before\\.|thanks\\.)$'
STOP = ['a', 'am', 'an', 'and', 'are', 'as', 'at', 'be', 'been', 'by', 'can', 'could', 'did', 'do', 'does', 'for', 'from', 'had', 'has', 'have', 'i', 'in', 'is', 'it', 'its', 'me', 'my', 'of', 'on', 'or', 'our', 'that', 'the', 'this', 'to', 'very', 'was', 'we', 'were', 'will', 'with', 'would', 'you', 'your']   # stop words: note that 'not' is NOT in this list

**PREDICT first.** How many words will a message have on average once the opener, closer, punctuation and stop words are gone: about 12, 5 or 1?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
clean = None  # lowercase, remove OPEN, remove CLOSE, then remove every character that is not a letter or space (r"[^a-z\s]")

In [ ]:
docs = None  # split each clean message into tokens and keep only tokens that are not in STOP

In [ ]:
try:
    vocab = sorted({w for d in docs for w in d})
    mean_tokens = float(np.mean([len(d) for d in docs]))
    n_texts = clean.nunique()
    print(len(vocab), 'different words;', round(mean_tokens, 1), 'tokens per message;', n_texts, 'different texts among', len(clean), 'messages')
    clean.head(3).tolist()
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
check('vocabulary size', safe(lambda: len(vocab)), 267, 0)
check('mean tokens per message', safe(lambda: mean_tokens), 5.3775, 0.01)
check('different texts after cleaning', safe(lambda: n_texts), 360, 0)

**READ the output.** The file has 800 messages but far fewer different texts. What does that mean for a model that is tested on messages it may have already seen? (We deal with this in step 5.)

> My answer: 

**CHANGE one thing.** Print `fb.message.str.lower().str.contains(r"\bnot\b").sum()` and then the same on `clean`. Why do the two counts differ?

> What I saw: 

---
## Step 3: Bag of words

Count each word of the vocabulary in each message. One row per message, one column per word. This is the document-term matrix.

**PREDICT first.** Which word is in the most messages? Guess before you run.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
X = None  # one row per message, one column per word: d.count(w)

In [ ]:
top_word = None  # the number of messages that contain each word is (X > 0).sum(axis=0); take the word with the largest count

In [ ]:
try:
    print(X.shape, X.sum(), top_word)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
check('columns of X', safe(lambda: X.shape[1]), 267, 0)
check('tokens counted in X', safe(lambda: X.sum()), 4302, 0)
check('most common word is phone', safe(lambda: top_word == 'phone'), True, 0)

**READ the output.** The top word is a filler about a phone number question. Why does it come first? (Hint: look at `fb.message.value_counts().head(3)`.)

> My answer: 

---
## Step 4: TF-IDF

Weight each count by how rare the word is: idf = ln(N / df), where df is the number of messages containing the word (the document frequency). Then scale each row to length 1.

In [ ]:
doc_freq = None  # document frequency: the number of messages in which each word appears, (X > 0).sum(axis=0)

In [ ]:
idf = None  # inverse document frequency: np.log(N / doc_freq) with N = len(fb)

In [ ]:
try:
    T = X * idf
    T = T / np.linalg.norm(T, axis=1, keepdims=True)   # every row now has length 1
    q = vocab.index('queue')
    print('queue: document frequency', doc_freq[q], 'idf', round(idf[q], 2))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
check('document frequency of queue', safe(lambda: doc_freq[vocab.index('queue')]), 19, 0)
check('idf of queue', safe(lambda: idf[vocab.index('queue')]), 3.7402, 0.001)

**CHANGE one thing.** Print the idf of 'costly' and of 'please'. Which is larger, and why?

> What I saw: 

---
## Step 5: Sentiment classification, scored on texts the model has not seen

Sentiment is the target. The TF-IDF columns are the features. Split by distinct text: all copies of one text go to the same fold (grouped 5-fold cross-validation), so every message is predicted by a model that never saw its text.

In [ ]:
CLASSES = ['negative', 'neutral', 'positive']
y = fb.sentiment.map({c: i for i, c in enumerate(CLASSES)}).values

def fit_softmax(X, y, k=3, lr=10.0, epochs=3000):
    # multinomial logistic regression by gradient descent (already written for you)
    # the penalty 1/len(X) on the weights is the same strength as scikit-learn's default C = 1
    l2 = 1.0 / len(X)
    W = np.zeros((X.shape[1], k)); b = np.zeros(k); Y = np.eye(k)[y]
    for _ in range(epochs):
        Z = X @ W + b; Z -= Z.max(axis=1, keepdims=True); P = np.exp(Z); P /= P.sum(axis=1, keepdims=True)
        G = (P - Y) / len(X); W -= lr * (X.T @ G + l2 * W); b -= lr * G.sum(axis=0)
    return W, b

def group_folds(group_id, k=5):
    # fold number for each message, all copies of one text in one fold (the idea inside scikit-learn's GroupKFold)
    sizes = np.bincount(group_id); order = np.argsort(sizes)[::-1]
    load = np.zeros(k); fold_of_group = np.zeros(len(sizes), dtype=int)
    for g in order:
        f = int(np.argmin(load)); load[f] += sizes[g]; fold_of_group[g] = f
    return fold_of_group[group_id]

In [ ]:
try:
    groups = [' '.join(d) for d in docs]                       # the cleaned text of each message
    _, group_id = np.unique(groups, return_inverse=True)   # one number per different text
    fold = group_folds(group_id, 5)
    print(group_id.max() + 1, 'different texts; messages per fold:', np.bincount(fold))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

**PREDICT first.** The baseline model says negative for every message. Will the classifier, scored on texts it has not seen, reach 60, 80 or 95 percent?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
def cv_predict(fold):
    return None   # for each of the 5 folds: test_rows = fold == f; fit_softmax on T[~test_rows]; predict T[test_rows]; store in an array of 800 integers

In [ ]:
try:
    oof = cv_predict(fold)   # oof = out-of-fold predictions
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
baseline = float((y == 0).mean())   # majority-class baseline: always say negative
accuracy = safe(lambda: float((oof == y).mean()))
fold_accuracies = safe(lambda: [round(float((oof[fold == f] == y[fold == f]).mean()), 3) for f in range(5)])
print('accuracy', accuracy, 'baseline', baseline, 'by fold', fold_accuracies)

In [ ]:
check('grouped 5-fold accuracy', accuracy, 0.774, 0.04)
check('baseline', baseline, 0.506, 0.001)

**READ the output.** About 6 percent of the labels are wrong on purpose, and the five fold scores differ. What does that say about trusting one split?

> My answer: 

**CHANGE one thing.** Now cut the folds at random instead (cell below) and run `cv_predict` again. Compare the accuracy with the grouped one, and count the test messages whose text was also in training.

> What I saw: 

In [ ]:
try:
    idx0 = np.random.RandomState(42).permutation(len(fb))
    fold_shuffled = np.empty(len(fb), dtype=int)
    fold_shuffled[idx0] = np.arange(len(fb)) // 160   # five random folds of 160 messages
    oof_shuffled = cv_predict(fold_shuffled)
    leak = 0
    for f in range(5):
        seen = set(group_id[fold_shuffled != f])
        leak += sum(g in seen for g in group_id[fold_shuffled == f])
    print('shuffled accuracy', float((oof_shuffled == y).mean()), '| test messages with a twin in training:', leak)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
check('shuffled 5-fold accuracy (too high)', safe(lambda: float((oof_shuffled == y).mean())), 0.9425, 0.03)
check('test messages with a twin in training', safe(lambda: leak), 565, 20)

---
## Step 6: Read the mistakes

A score is a summary. Read the wrong answers and decide for each: bad label or model error?

In [ ]:
mistakes = None  # rows of fb where oof differs from y

In [ ]:
try:
    n_wrong = len(mistakes)
    print(n_wrong, 'wrong answers')
    mistakes[['message', 'sentiment', 'model']].drop_duplicates('message').head(11)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
check('wrong answers', safe(lambda: n_wrong), 181, 8)

**READ the output.** How many of the wrong answers look like a bad label (the message and the label disagree plainly)? Write the number and one example.

> My answer: 

---
## Step 7: Where counting words fails: negation

A bag of words ignores word order. Fit one final model on all 800 messages and test two new sentences.

In [ ]:
W, b = None, None  # fit_softmax(T, y)

In [ ]:
STOPSET = set(STOP)
def tokens(text):
    t = re.sub(CLOSE, '', re.sub(OPEN, '', text.lower()))
    return [w for w in re.sub(r'[^a-z\s]', '', t).split() if w not in STOPSET]

def predict_proba(text):
    v = np.zeros(len(vocab))
    for w in tokens(text):
        if w in vocab:
            v[vocab.index(w)] += 1
    v = v * idf
    v = v / np.linalg.norm(v)
    z = v @ W + b
    p = np.exp(z - z.max())
    return p / p.sum()

**PREDICT first.** Will the model call "the queue was not good" negative or positive?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
p_good = None  # predict_proba('the queue was good')[2]  (index 2 is positive)

In [ ]:
p_notgood = None  # predict_proba('the queue was not good')[2]

In [ ]:
print(safe(lambda: round(p_good, 2)), safe(lambda: round(p_notgood, 2)))
check('probability of positive: the queue was good', safe(lambda: p_good), 0.939, 0.05)
check('probability of positive: the queue was not good', safe(lambda: p_notgood), 0.717, 0.05)

**CHANGE one thing.** Try "the queue was not bad" and "the queue was terrible". Which does the model get right, and which words does it rely on?

> What I saw: 

---
## Step 8: The same recipe with scikit-learn (the tools used at work)

CountVectorizer builds the bag of words. TfidfVectorizer builds TF-IDF. make_pipeline chains the vectoriser and a classifier so the vocabulary and idf are learned from the training folds only.

In [ ]:
try:
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report
    from sklearn.model_selection import GroupKFold, cross_val_predict
    from sklearn.pipeline import make_pipeline
    HAVE_SK = True
except ImportError:
    HAVE_SK = False
    print('scikit-learn is not installed here. Run this step in Colab.')

In [ ]:
if HAVE_SK:
    texts = [' '.join(d) for d in docs] if docs is not None else []   # the cleaned text of each message

**PREDICT first.** The vocabulary of CountVectorizer should match your vocab. Will its idf for queue match the idf you computed in step 4? Guess, then run.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
if HAVE_SK:
    counts = None  # CountVectorizer().fit_transform(texts)
    tfidf = None   # TfidfVectorizer().fit(texts)
    idf_queue_sk = None   # tfidf.idf_[tfidf.vocabulary_['queue']]

In [ ]:
if HAVE_SK:
    check('scikit-learn vocabulary size', safe(lambda: counts.shape[1]), 267, 0)
    check('scikit-learn idf of queue (smoothed formula)', safe(lambda: idf_queue_sk), 4.690, 0.02)

**READ the output.** scikit-learn adds 1 inside the logarithm and 1 outside: idf = ln((1 + N) / (1 + df)) + 1. Why is its idf for queue larger than yours?

> My answer: 

In [ ]:
if HAVE_SK:
    pipe = None     # make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
    pred_sk = None  # cross_val_predict(pipe, texts, y, groups=group_id, cv=GroupKFold(n_splits=5))

In [ ]:
if HAVE_SK:
    accuracy_sk = safe(lambda: float((pred_sk == y).mean()))
    print('accuracy', accuracy_sk)
    safe(lambda: print(classification_report(y, pred_sk, target_names=CLASSES)))
    check('scikit-learn pipeline, grouped accuracy', accuracy_sk, 0.789, 0.05)

**CHANGE one thing.** Replace LogisticRegression with another classifier from the Session 17 family and compare the grouped accuracy. Which wins, and by how much?

> What I saw: 

---
## Step 9: Score a keyword search with the answer key

The theme variable says what each message is about, so it is an answer key. Search one row per different text, so ten copies of one sentence cannot fill the results.

In [ ]:
try:
    first_row = np.array([np.where(group_id == g)[0][0] for g in range(group_id.max() + 1)])
    ufb = fb.iloc[first_row].reset_index(drop=True)      # one row per different text
    udocs = [docs[i] for i in first_row]
    uT = T[first_row]
    print(len(ufb), 'different texts to search')
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
is_queue = None  # a True/False array: ufb.theme == 'queue' (use .values)

In [ ]:
has_word = None  # a True/False array: does 'queue' appear in the tokens of each text in udocs

In [ ]:
n_found = None  # the number of texts that are both queue texts and contain the word

In [ ]:
recall = None  # n_found divided by the number of queue texts: recall

In [ ]:
print(safe(lambda: n_found), 'of', safe(lambda: is_queue.sum()), 'found; recall', safe(lambda: round(recall, 3)))
check('queue texts found', safe(lambda: n_found), 10, 0)
check('recall of the keyword search', safe(lambda: recall), 0.2778, 0.005)

**CHANGE one thing.** Search for 'queue' or 'line' (use `any(k in d for k in ['queue', 'line'])`). What happens to recall and to precision?

> What I saw: 

---
## Step 10: Cosine similarity and TF-IDF search

Cosine similarity: the dot product of two vectors divided by the product of their lengths.

In [ ]:
def cosine(a, b):
    return None  # a @ b divided by (length of a times length of b)

In [ ]:
c_ab = cosine(np.array([3, 4]), np.array([4, 3]))
print(c_ab)
check('cosine of (3,4) and (4,3)', safe(lambda: c_ab), 0.96, 0.001)

In [ ]:
def query_vector(text):
    v = np.zeros(len(vocab))
    for w in tokens(text):
        if w in vocab:
            v[vocab.index(w)] += 1
    v = v * idf
    return v / np.linalg.norm(v)

def precision_at_10(text):
    scores = uT @ query_vector(text)         # rows of uT have length 1, so this is the cosine
    top = np.argsort(-scores, kind='stable')[:10]
    return float((ufb.theme.values[top] == 'queue').mean())

**PREDICT first.** Which query finds more queue texts in its top 10: "many people waiting to pay" or "crowded cashier"?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
p10_ok = None  # precision_at_10 of the first query

In [ ]:
p10_bad = None  # precision_at_10 of the second query

In [ ]:
print(p10_ok, p10_bad)
check('precision at 10, many people waiting to pay', safe(lambda: p10_ok), 0.7, 0.05)
check('precision at 10, crowded cashier', safe(lambda: p10_bad), 0.4, 0.05)

**READ the output.** The word crowded is not in the vocabulary. Which words did the second query actually use, and what are the top 3 texts about?

> My answer: 

---
## Step 11: Semantic search with embeddings (needs sentence-transformers, free in Colab)

An embedding model turns each text into a vector of 384 numbers. Then the same cosine search ranks by meaning.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    try:
        import sentence_transformers
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'])
try:
    from sentence_transformers import SentenceTransformer
    HAVE_ST = True
except ImportError:
    HAVE_ST = False
    print('sentence-transformers is not installed here. Run this step in Colab.')

**PREDICT first.** Will "crowded cashier" now find more queue texts than in step 10? Write your guess, then run.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
if HAVE_ST:
    model_st = SentenceTransformer('all-MiniLM-L6-v2')   # check its model card for size and licence
    emb = None  # model_st.encode(list of texts, normalize_embeddings=True)
    check('embedding columns (384)', safe(lambda: emb.shape[1]), 384, 0)
    check('embedding rows (one per different text)', safe(lambda: emb.shape[0]), len(ufb), 0)
    for text in ['many people waiting to pay', 'crowded cashier']:
        try:
            scores = emb @ model_st.encode([text], normalize_embeddings=True)[0]
            top = np.argsort(-scores, kind='stable')[:10]
            print(text, '-> precision at 10:', float((ufb.theme.values[top] == 'queue').mean()))
        except Exception:
            print(text, '-> not ready yet')

**READ the output.** Write your two precision-at-10 numbers from this step and compare them with step 10. Which query changed, and why? (No number is printed in the slides because it was not run when the slides were written.)

> My answer: 

---
## Step 12: Keep: a scored file and a three-sentence finding

Save every message with the prediction a model made without seeing its text. Then write your finding in the cell below.

In [ ]:
all_pred = None  # the out-of-fold predictions from step 5 (oof)

In [ ]:
try:
    scored = fb[['feedback_id', 'message', 'theme', 'sentiment']].copy()
    scored['predicted_sentiment'] = [CLASSES[i] for i in all_pred]
    scored['correct'] = scored.sentiment == scored.predicted_sentiment
    scored.to_csv('keep/feedback_scored.csv', index=False)
    print(len(scored), 'rows saved in the folder keep; accuracy on all 800 (grouped):', round(float(scored.correct.mean()), 3))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
check('rows in feedback_scored.csv', safe(lambda: len(scored)), 800, 0)
check('accuracy on all 800 messages (grouped predictions)', safe(lambda: scored.correct.mean()), 0.774, 0.04)
show_checks()

**My finding (three sentences):** 1) the grouped accuracy and the baseline, 2) one mistake I read and who was wrong, 3) one search the keyword method missed.

> 